In [130]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from utils.MultiLabelPredictor import MultilabelPredictor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [142]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
class PredictExposure:
    def __init__(self, model):
        self.model = model 

    def fit(self, mutation_count, additional_features=None):
        if additional_features is not None:
            mutation_count = np.hstack([mutation_count, additional_features])

        mutation_count_bin = pd.read_csv('../simulations/ground_truth/bin_exposures.csv').iloc[:,1:].astype(int).values  # N x 29
        signature_exposure = pd.read_csv('../simulations/ground_truth/exposures.csv').iloc[:,1:].values  # N x 29

        X_train, X_test, y_bin_train, y_bin_test, y_exp_train, y_exp_test = train_test_split(
            mutation_count, mutation_count_bin, signature_exposure, train_size=0.8, random_state=42
        )
        regressors = []
        r2_scores = []
        mse_scores = []

        for i in range(y_bin_train.shape[1]):  # ciclo sulle 29 firme
            # Filtra campioni con firma attiva in train
            idx_train = y_bin_train[:, i] == 1
            X_train_i = X_train[idx_train]
            y_train_i = y_exp_train[idx_train, i]

            # Filtra campioni con firma attiva in test
            idx_test = y_bin_test[:, i] == 1
            X_test_i = X_test[idx_test]
            y_test_i = y_exp_test[idx_test, i]

            # Se non ci sono campioni attivi in train o test, skip
            if len(y_train_i) == 0 or len(y_test_i) == 0:
                regressors.append(None)
                r2_scores.append(np.nan)
                mse_scores.append(np.nan)
                continue

            # Addestra regressore lineare
            self.model.fit(X_train_i, y_train_i)
            regressors.append(self.model)

            # Predici su test
            y_pred_i = self.model.predict(X_test_i)

            # Valuta performance
            r2_scores.append(r2_score(y_test_i, y_pred_i))
            mse_scores.append(mean_squared_error(y_test_i, y_pred_i))

        # # Output risultati
        # for i in range(len(regressors)):
        #     print(f"Signature_{i+1}: R2 = {r2_scores[i]:.3f}, MSE = {mse_scores[i]:.3e}")
        return r2_scores, mse_scores

In [153]:
def compare_r2_sets(s1, s2):
    s1 = np.array(s1)
    s2 = np.array(s2)
    mean1, mean2 = np.nanmean(s1), np.nanmean(s2)
    median1, median2 = np.nanmedian(s1), np.nanmedian(s2)
    better_in_s2 = (s2 > s1).sum()
    print(f"Mean R² set1: {mean1:.3f}")
    print(f"Mean R² set2: {mean2:.3f}")
    print(f"Median R² set1: {median1:.3f}")
    print(f"Median R² set2: {median2:.3f}")
    print(f"Set2 better in {better_in_s2} / {len(s1)} signatures")
    if mean2 > mean1:
        print("Set2 has better overall R²")
    elif mean2 < mean1:
        print("Set1 has better overall R²")
    else:
        print("Both sets have equal mean R²")


In [144]:
predictor = MultilabelPredictor.load('../models/save/Predictior-0.03')
pred_mutation_count = pd.read_csv('../simulations/data2/run_1/trinucleotides_counts_sampling_0.03.csv').iloc[:,1:]
pred_mutation_count.head()
#predictions = predictor.predict(pred_mutation_count)
#predictions

,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,A[C>T]C,...,T[T>A]G,T[T>A]T,T[T>C]A,T[T>C]C,T[T>C]G,T[T>C]T,T[T>G]A,T[T>G]C,T[T>G]G,T[T>G]T
0,9,14,2,15,2,1,0,1,6,2,...,3,3,3,4,1,3,1,3,4,0
1,0,2,0,2,0,0,0,1,3,2,...,0,0,0,0,1,2,0,0,0,2
2,0,1,0,3,0,0,0,0,0,4,...,1,2,3,0,1,1,2,0,0,0
3,2,2,0,3,0,1,0,0,5,0,...,2,3,3,1,1,3,2,5,0,2
4,0,2,0,1,0,0,0,0,0,1,...,0,1,2,0,1,1,0,0,0,1


### Train's Data

In [145]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Caricamento dati
signature_prob_distribution = pd.read_csv('../simulations/ground_truth/signatures.csv').iloc[:,1:].values
mutation_count = pd.read_csv('../simulations/data/run_1/trinucleotides_counts_sampling_0.03.csv').iloc[:,1:].values  # N x 96

### Base with Linear regressioin and no features engeneering

In [146]:
model = LinearRegression()
model_mutational_count = mutation_count @ signature_prob_distribution.T

pe = PredictExposure(model= model)
start_r2, start_mse = pe.fit(model_mutational_count)

### Mutational count normalization and Linear Regression

In [155]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# oppure: standardizzazione
scaler = StandardScaler()
mutation_scaled = scaler.fit_transform(mutation_count)
model_mutational_count = mutation_scaled @ signature_prob_distribution.T
r2_1, mse_1 = pe.fit(model_mutational_count)

compare_r2_sets(start_r2, r2_1)

Mean R² set1: -1.515
Mean R² set2: -7.076
Median R² set1: 0.888
Median R² set2: 0.879
Set2 better in 13 / 29 signatures
Set1 has better overall R²


### Concatenate Bin vectors

In [156]:
model = LinearRegression()
predictor = PredictExposure(model)
mutation_count_proj = mutation_count @ signature_prob_distribution.T  # N x 29 (stima esposizioni)
r2_2, mse_2 = predictor.fit(mutation_count_proj, additional_features=mutation_count)  # input arricchito

compare_r2_sets(start_r2, r2_2)

Mean R² set1: -1.515
Mean R² set2: -1.030
Median R² set1: 0.888
Median R² set2: 0.842
Set2 better in 11 / 29 signatures
Set2 has better overall R²


### Utilizzo di un modello neurale

In [157]:
from tensorflow.keras import layers, models

class KerasRegressorWrapper:
    def __init__(self, input_dim):
        self.model = self.build_model(input_dim)

    def build_model(self, input_dim):
        model = models.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(64, activation='relu'),
            layers.Dense(32, activation='relu'),
            layers.Dense(1, activation='linear')
        ])
        model.compile(optimizer='adam', loss='mse')
        return model

    def fit(self, X, y):
        self.model.fit(X, y, epochs=50, batch_size=32, verbose=0)

    def predict(self, X):
        return self.model.predict(X).flatten()


ModuleNotFoundError: No module named 'tensorflow'